<a href="https://colab.research.google.com/github/priyalpatil31/AML_Priyal_Patil_19/blob/main/EXP_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from mlxtend.feature_selection import SequentialFeatureSelector

path = "/content/drive/MyDrive/Online_Retail_Small.xlsx"

df = pd.read_excel(path)

df = df.fillna(df.mean(numeric_only=True))

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])
    df[col] = df[col].astype(str)

for col in df.columns:
    if "date" in col.lower():
        df.drop(col, axis=1, inplace=True)

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('Quantity', axis=1)
y = df['Quantity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()

print("SEQUENTIAL FORWARD SELECTION")

sfs = SequentialFeatureSelector(
    model,
    k_features=5,
    forward=True,
    floating=False,
    scoring='r2',
    cv=5
)

sfs.fit(X_train, y_train)

forward_features = list(sfs.k_feature_names_)

print(forward_features)

model.fit(X_train[forward_features], y_train)

y_pred1 = model.predict(X_test[forward_features])

score1 = r2_score(y_test, y_pred1)

print("\nSFS Model Score :", score1)

print("\nSEQUENTIAL BACKWARD ELIMINATION")

sbe = SequentialFeatureSelector(
    model,
    k_features=5,
    forward=False,
    floating=False,
    scoring='r2',
    cv=5
)

sbe.fit(X_train, y_train)

backward_features = list(sbe.k_feature_names_)

print(backward_features)

model.fit(X_train[backward_features], y_train)

y_pred2 = model.predict(X_test[backward_features])

score2 = r2_score(y_test, y_pred2)

print("\nSBE Model Score :", score2)

print("\nCOMPARISON")

if score1 > score2:
    print("SFS gives better performance")
else:
    print("SBE gives better performance")

Mounted at /content/drive
SEQUENTIAL FORWARD SELECTION
['InvoiceNo', 'StockCode', 'UnitPrice', 'CustomerID', 'Country']

SFS Model Score : -0.31273219312543166

SEQUENTIAL BACKWARD ELIMINATION
['InvoiceNo', 'StockCode', 'UnitPrice', 'CustomerID', 'Country']

SBE Model Score : -0.31273219312543166

COMPARISON
SBE gives better performance
